# CSV - Missing Values

Add scattered, grouped, or mixed null values to a selected column.

**Instructions:** Use `prepare_data_csv_update.ipynb` to generate a clean `data/weather.csv` first. The notebook overwrites it; restore clean data before rerunning.


In [ ]:
from pathlib import Path
from random import Random
import pandas as pd

# Configuration
DATA_PATH = Path("data/weather.csv")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local calendar dates.
SEED = 814205

# Missing Values
MISSING_TARGET_COLUMN = "relative_humidity_2m"
MISSING_METHOD = "mixed"  # "scattered", "grouped", "mixed", "random".
MISSING_RATE = (0.03, 0.08)  # Share of current eligible nonmissing values changed.
MISSING_GROUPS = 3
MISSING_GROUPED_SHARE = 0.70


In [ ]:
# Shared placement helper: copy once into a combined dirty_data.py.
def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen

In [ ]:
def add_missing_values(
    df, rng, *, time_column, timezone, start_date, end_date,
    target_column, method, missing_rate, n_groups, grouped_share,
):
    # Select eligible records in chronological order (not necessarily hourly).
    if time_column not in df or target_column not in df:
        raise ValueError("Time or target column not found.")

    dirty = df.reset_index(drop=True).copy()
    time = pd.to_datetime(dirty[time_column], utc=True)
    if time.isna().any():
        raise ValueError("Timestamps cannot be missing.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")

    scope = time.ge(start) & time.lt(end)
    eligible = time.loc[scope].sort_values(kind="stable")
    if eligible.empty:
        raise ValueError("No eligible observations.")
    if dirty.loc[scope, target_column].isna().any():
        raise ValueError("Target already contains missing values in this interval. Restore clean data first.")

    # Rate refers to currently eligible records in this configured interval.
    selected, _ = select_positions(
        len(eligible), rng, method=method, rate=missing_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    dirty.loc[eligible.index[selected], target_column] = pd.NA
    return dirty


In [ ]:
# Small trial: preserve row count and non-target fields.
trial_time = pd.date_range("2025-08-01", periods=7 * 24, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = pd.DataFrame({TIME_COLUMN: trial_time, MISSING_TARGET_COLUMN: range(len(trial_time)), "other": 1})
trial_dirty = add_missing_values(
    trial, Random(SEED), time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date="2025-08-01", end_date="2025-08-08",
    target_column=MISSING_TARGET_COLUMN, method="mixed", missing_rate=(0.10, 0.10),
    n_groups=2, grouped_share=0.70,
)
assert len(trial_dirty) == len(trial)
assert trial_dirty[MISSING_TARGET_COLUMN].isna().sum() == round(len(trial) * 0.10)
pd.testing.assert_frame_equal(trial_dirty.drop(columns=MISSING_TARGET_COLUMN), trial.drop(columns=MISSING_TARGET_COLUMN))
trial_dirty.loc[trial_dirty[MISSING_TARGET_COLUMN].isna()].head()


In [ ]:
# Apply to a freshly prepared CSV. This overwrites weather.csv.
clean = pd.read_csv(DATA_PATH)
rng = Random(SEED)
dirty = add_missing_values(
    clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    target_column=MISSING_TARGET_COLUMN, method=MISSING_METHOD,
    missing_rate=MISSING_RATE, n_groups=MISSING_GROUPS,
    grouped_share=MISSING_GROUPED_SHARE,
)
dirty.to_csv(DATA_PATH, index=False)
saved = pd.read_csv(DATA_PATH)


In [ ]:
saved.isna().any().any()

In [ ]:
saved.isna().sum()

In [ ]:
saved.loc[saved.isna().any(axis=1)].head(20)

In [ ]:
# Verify introduced nulls and preserved data after CSV round-trip.
new_nulls = saved[MISSING_TARGET_COLUMN].isna() & clean[MISSING_TARGET_COLUMN].notna()
assert new_nulls.any()
assert len(saved) == len(clean)
pd.testing.assert_frame_equal(saved.drop(columns=MISSING_TARGET_COLUMN), clean.drop(columns=MISSING_TARGET_COLUMN), check_exact=False, rtol=1e-12)
pd.testing.assert_series_equal(saved.loc[~new_nulls, MISSING_TARGET_COLUMN], clean.loc[~new_nulls, MISSING_TARGET_COLUMN], check_dtype=False, check_exact=False, rtol=1e-12)
print("New missing values:", int(new_nulls.sum()))

local_dates = pd.to_datetime(clean[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
comparison = pd.DataFrame({
    "clean_mean": clean.groupby(local_dates)[MISSING_TARGET_COLUMN].mean(),
    "dirty_mean": saved.groupby(local_dates)[MISSING_TARGET_COLUMN].mean(),
    "clean_count": clean.groupby(local_dates)[MISSING_TARGET_COLUMN].count(),
    "dirty_count": saved.groupby(local_dates)[MISSING_TARGET_COLUMN].count(),
})
comparison.loc[comparison["clean_count"].ne(comparison["dirty_count"])].head(30)


In [ ]:
import matplotlib.pyplot as plt

# Date × hour missingness map for the configured interval.
time = pd.to_datetime(saved[TIME_COLUMN], utc=True)
start = pd.Timestamp(START_DATE, tz=TIMEZONE).tz_convert("UTC")
end = pd.Timestamp(END_DATE, tz=TIMEZONE).tz_convert("UTC")
view = saved.loc[time.ge(start) & time.lt(end), [TIME_COLUMN, MISSING_TARGET_COLUMN]].copy()
local = pd.to_datetime(view[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE)
view["date"] = local.dt.date
view["hour"] = local.dt.hour
view["missing"] = view[MISSING_TARGET_COLUMN].isna().astype(int)
pattern = (view.pivot_table(index="date", columns="hour", values="missing", aggfunc="max", fill_value=0)
           .reindex(columns=range(24), fill_value=0))

fig, ax = plt.subplots(figsize=(10, max(3, len(pattern) * 0.16)))
ax.imshow(pattern, aspect="auto", interpolation="nearest", cmap="Greys", vmin=0, vmax=1)
ax.set_xticks([0, 6, 12, 18, 23])
ax.set_xlabel("Local hour")
ax.set_ylabel("Date")
ax.set_title(f"Missing values: {MISSING_TARGET_COLUMN}")
step = max(1, len(pattern) // 8)
ax.set_yticks(range(0, len(pattern), step), [str(d) for d in pattern.index[::step]])
plt.show()
